# 🐾 Checkpoint — Monitoramento da Fauna

**Parque nacional de Cerrado · 12 câmeras automáticas · 01 a 30/05/2026 · 5.000 registros**

Este notebook analisa os registros brutos das câmeras do programa de monitoramento da fauna, respondendo aos desafios propostos no README e atuando como **analista de dados** da equipe.

**Estrutura do dataset** (1 registro = 1 detecção de fauna pelas câmeras):

| Coluna | Descrição |
|---|---|
| `id_registro` | Identificação do registro |
| `id_camera` | Câmera que fez o registro (CAM01 a CAM12, distribuídas em 3 áreas) |
| `data_hora_inicio` | Data e hora em que o registro ocorreu |
| `duracao_segundos` | Duração do registro em segundos |
| `especie` / `grupo` | Espécie identificada e grupo faunístico (Mamífero, Ave, Anfíbio) |
| `individuos` | Quantidade de indivíduos no registro |
| `temperatura_c` / `umidade_pct` | Leitura dos sensores no momento do registro |

**Roteiro do notebook:**
1. Preparação dos dados (Bônus 1)
2. Desafios 1 a 5 (capivaras, períodos do dia, anfíbios x umidade, câmeras e áreas, onça-pintada)
3. Bônus 2, 3 e 4 (duração dos registros, lobo-guará, investigação própria)
4. Conclusões gerais para a apresentação

In [0]:
# ============================================================
# IMPORTAÇÕES E CARREGAMENTO DOS DADOS
# ============================================================
# Bibliotecas usadas na análise:
# - json: ler o arquivo dataset.json (lista de registros em texto estruturado)
# - re: expressões regulares, para extrair números de textos (ex.: "19.3 °C")
# - pandas: manipular os dados em tabela (DataFrame)
# - numpy: valores numéricos e ausentes (NaN)
# - matplotlib.pyplot: gráficos para visualizar os resultados
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Caminho do arquivo de dados fornecido pelo checkpoint
CAMINHO_DADOS = "/Workspace/Repos/cardosoamelia2001@gmail.com/fast-track-ia-t1/checkpoints/monitoramento-fauna/dataset.json"

# Lendo o JSON: json.load() converte o arquivo em uma lista de dicionários
# Python; o pandas transforma essa lista em uma tabela (DataFrame)
with open(CAMINHO_DADOS, "r", encoding="utf-8") as arquivo:
    registros = json.load(arquivo)

df = pd.DataFrame(registros)

# Visualizando as primeiras linhas para conhecer a estrutura dos dados
display(df.head())

## 🧹 Bônus 1 — Preparando os dados: diagnóstico inicial

Antes de qualquer análise, verificamos a **qualidade dos dados**:

* **Estrutura**: quantos registros e colunas existem;
* **Tipos**: se números estão armazenados como números (e não como texto);
* **Valores ausentes**: se alguma leitura de sensor faltou;
* **Duplicidades**: se o mesmo registro aparece mais de uma vez;
* **Categorias**: se câmeras, espécies e grupos batem com o esperado no README (12 câmeras, 13 espécies, 3 grupos) e se cada espécie pertence a um único grupo.

In [0]:
# ============================================================
# BÔNUS 1 — DIAGNÓSTICO INICIAL DOS DADOS
# ============================================================
# 1) Quantas linhas e colunas existem?
print(f"Dimensões do dataset: {df.shape[0]} registros x {df.shape[1]} colunas")

# 2) Quais os tipos de cada coluna?
# 'object' significa TEXTO: temperatura e umidade estão como texto, não número.
print("\nTipos de dados por coluna:")
print(df.dtypes)

# 3) Existem valores ausentes (nulos)?
print("\nValores ausentes por coluna:")
print(df.isna().sum())

# 4) Existem duplicatas (mesmo id_registro, ou linha inteira repetida)?
print(f"\nIDs de registro duplicados: {df['id_registro'].duplicated().sum()}")
print(f"Linhas completamente duplicadas: {df.duplicated().sum()}")

# 5) As categorias batem com o esperado no README?
print(f"\nCâmeras ({df['id_camera'].nunique()}): {sorted(df['id_camera'].unique())}")
print(f"\nEspécies ({df['especie'].nunique()}):")
print(df["especie"].value_counts())
print(f"\nGrupos: {sorted(df['grupo'].unique())}")

# Cada espécie aparece com um único grupo? (nunique > 1 indicaria inconsistência)
grupos_por_especie = df.groupby("especie")["grupo"].nunique()
print(f"\nEspécies com mais de um grupo (deveria ser vazio): {grupos_por_especie[grupos_por_especie > 1].to_dict()}")

## 🧹 Bônus 1 — Decisões de tratamento

Com base no diagnóstico, os problemas encontrados e as decisões foram:

| Problema encontrado | Decisão | Justificativa |
|---|---|---|
| `temperatura_c` e `umidade_pct` são **texto** com unidades (ex.: `"19.3 °C"`, `"81.5%"`) | Extrair apenas o número (expressão regular) e converter para `float` | Não é possível calcular médias, faixas ou comparações com texto |
| `data_hora_inicio` é **texto** | Converter para o tipo `datetime` | Permite extrair a hora e analisar a atividade ao longo do dia |
| **247 registros sem temperatura** e **175 sem umidade** | Manter como ausentes (`NaN`), sem preencher com média | Preencher (imputar) poderia distorcer análises como o Desafio 3, que depende da umidade real medida no momento; cada análise usa apenas os registros com a leitura disponível |
| Sem duplicatas; câmeras, espécies e grupos consistentes com o README | Nada a corrigir | Dados já válidos |

Também criamos **colunas derivadas** usadas nas análises: `hora` (0–23), `periodo` (Madrugada/Manhã/Tarde/Noite) e `area` (área do parque de cada câmera, conforme o README).

In [0]:
# ============================================================
# BÔNUS 1 — LIMPEZA E PREPARAÇÃO DOS DADOS
# ============================================================
# PROBLEMA 1: temperatura_c e umidade_pct vêm como TEXTO com unidades
# (ex.: "19.3 °C", "81.5%"). Não dá para calcular com texto.
# DECISÃO: extrair apenas o número (expressão regular) e converter
# para float, criando colunas numéricas no lugar das antigas.
def extrair_numero(valor):
    """Extrai o número de um texto como '19.3 °C' ou '81.5%'.
    Se o valor for nulo, devolve NaN (ausente) — sem inventar dado."""
    if pd.isna(valor):
        return np.nan
    numero = re.search(r"-?\d+(?:\.\d+)?", str(valor))  # procura o padrão numérico
    return float(numero.group(0)) if numero else np.nan

df["temperatura_c"] = df["temperatura_c"].apply(extrair_numero)
df["umidade_pct"] = df["umidade_pct"].apply(extrair_numero)

# PROBLEMA 2: data_hora_inicio está como texto.
# DECISÃO: converter para datetime, que permite extrair hora, dia etc.
df["data_hora_inicio"] = pd.to_datetime(df["data_hora_inicio"])

# PROBLEMA 3: 247 registros sem temperatura e 175 sem umidade.
# DECISÃO: manter como NaN (ausente), sem imputar valores.
# Cada análise informa e usa apenas os registros com leitura disponível.

# ---------- COLUNAS DERIVADAS (enriquecimento) ----------
# 1) 'hora' (0-23): extraída da data, para estudar períodos do dia
df["hora"] = df["data_hora_inicio"].dt.hour

# 2) 'periodo': classificação da hora em 4 faixas do dia
def classificar_periodo(hora):
    if 0 <= hora < 6:
        return "Madrugada"
    elif 6 <= hora < 12:
        return "Manhã"
    elif 12 <= hora < 18:
        return "Tarde"
    else:
        return "Noite"

df["periodo"] = df["hora"].apply(classificar_periodo)

# 3) 'area': cada câmera pertence a uma área do parque (conforme o README)
AREA_CAMERAS = {}
for cam in ["CAM01", "CAM02", "CAM03", "CAM04"]:
    AREA_CAMERAS[cam] = "Cerrado aberto"
for cam in ["CAM05", "CAM06", "CAM07", "CAM08"]:
    AREA_CAMERAS[cam] = "Mata de galeria"
for cam in ["CAM09", "CAM10", "CAM11", "CAM12"]:
    AREA_CAMERAS[cam] = "Veredas e áreas úmidas"

df["area"] = df["id_camera"].map(AREA_CAMERAS)

# Conferindo o resultado do tratamento
display(df.head())
print("\nTipos após o tratamento:")
print(df.dtypes)

## 🦫 Desafio 1 — Capivaras: solitárias ou em grupo?

**Pergunta:** qual o número médio de indivíduos por registro de capivara? A maioria dos registros é de indivíduos isolados ou de grupos?

**Abordagem:** filtramos os registros da espécie Capivara e calculamos (1) a média de `individuos` por registro, (2) a proporção de registros com 1 indivíduo (solitário) versus 2 ou mais (grupo) e (3) a distribuição completa da quantidade de indivíduos por registro.

In [0]:
# ============================================================
# DESAFIO 1 — CAPIVARAS: SOLITÁRIAS OU EM GRUPO?
# ============================================================
# Filtrando apenas os registros da espécie Capivara
capivaras = df[df["especie"] == "Capivara"].copy()

# 1) Número MÉDIO de indivíduos por registro
media_individuos = capivaras["individuos"].mean()

# 2) Registros solitários (1 indivíduo) x registros em grupo (2 ou mais)
solitarios = (capivaras["individuos"] == 1).sum()
grupos = (capivaras["individuos"] >= 2).sum()

print(f"Registros de capivara: {len(capivaras)}")
print(f"Média de indivíduos por registro: {media_individuos:.2f}")
print(f"Registros solitários (1 indivíduo): {solitarios} ({solitarios / len(capivaras) * 100:.1f}%)")
print(f"Registros em grupo (2+ indivíduos): {grupos} ({grupos / len(capivaras) * 100:.1f}%)")

# 3) Distribuição completa: quantos registros com 1, 2, 3... indivíduos
distribuicao = capivaras["individuos"].value_counts().sort_index()
print("\nDistribuição de indivíduos por registro:")
print(distribuicao)

# Gráfico de barras para visualizar a distribuição
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(distribuicao.index, distribuicao.values, color="#4c9f70")
ax.axvline(media_individuos, color="crimson", linestyle="--", label=f"Média = {media_individuos:.2f}")
ax.set_xlabel("Indivíduos no registro")
ax.set_ylabel("Quantidade de registros")
ax.set_title("Desafio 1 — Indivíduos por registro de capivara")
ax.legend()
plt.show()

### ✅ Resposta — Desafio 1

* **Média de indivíduos por registro de capivara: ≈ 5,95** (731 registros no mês).
* **A grande maioria dos registros é de grupos:** apenas **6 registros (0,8%)** são solitários; **725 (99,2%)** têm 2 ou mais indivíduos.
* A distribuição tem pico em **5 indivíduos** por registro (129 registros), com cauda até 14 indivíduos.

**Conclusão:** a capivara é registrada quase sempre **em grupo**, com média próxima de **6 indivíduos por registro** — coerente com o comportamento real da espécie, que vive em grupos familiares próximos à água.

## 🌅 Desafio 2 — Quando a fauna está mais ativa?

**Pergunta:** em quais períodos do dia há mais registros de animais? E os horários de maior ocorrência de três espécies, como se comparam?

**Abordagem:** usamos a coluna derivada `periodo` (Madrugada 0–6h, Manhã 6–12h, Tarde 12–18h, Noite 18–24h) e a `hora` exata. Primeiro a visão geral do parque; depois comparamos três espécies com papéis ecológicos distintos: **Capivara** (mamífero herbívoro de áreas úmidas), **Sapo-cururu** (anfíbio) e **Lobo-guará** (mamífero de hábitos crepusculares).

In [0]:
# ============================================================
# DESAFIO 2 — VISÃO GERAL: REGISTROS POR PERÍODO DO DIA
# ============================================================
# A coluna 'periodo' (criada na preparação) classifica cada registro em
# Madrugada, Manhã, Tarde ou Noite. Contamos os registros de cada período.
# reindex garante a ordem cronológica correta na exibição.
registros_por_periodo = df["periodo"].value_counts().reindex(["Madrugada", "Manhã", "Tarde", "Noite"])
print("Registros por período do dia (todas as espécies):")
print(registros_por_periodo)

# Gráfico de barras: registros por hora do dia (visão mais detalhada
# que a agregação por período — mostra exatamente onde estão os picos)
registros_por_hora = df["hora"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(registros_por_hora.index, registros_por_hora.values, color="#4a7fa5")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Quantidade de registros")
ax.set_title("Desafio 2 — Registros por hora do dia (todas as espécies)")
plt.show()

In [0]:
# ============================================================
# DESAFIO 2 — COMPARANDO OS HORÁRIOS DE 3 ESPÉCIES
# ============================================================
# Espécies escolhidas (papéis ecológicos distintos):
# - Capivara: mamífero herbívoro, típico de áreas úmidas
# - Sapo-cururu: anfíbio (atividade associada à umidade noturna)
# - Lobo-guará: mamífero onívoro, hábitos crepusculares
ESPECIES_ESCOLHIDAS = ["Capivara", "Sapo-cururu", "Lobo-guará"]

# Tabela cruzada (crosstab): quantos registros de cada espécie em cada período
subconjunto = df[df["especie"].isin(ESPECIES_ESCOLHIDAS)]
comparacao = pd.crosstab(subconjunto["especie"], subconjunto["periodo"])
comparacao = comparacao[["Madrugada", "Manhã", "Tarde", "Noite"]]  # ordem cronológica
print("Registros por período e espécie:")
display(comparacao)

# O período e as horas com MAIS registros de cada espécie
for especie in ESPECIES_ESCOLHIDAS:
    registros_especie = df[df["especie"] == especie]
    periodo_pico = registros_especie["periodo"].value_counts().idxmax()
    horas_pico = list(registros_especie["hora"].value_counts().head(3).index)
    print(f"{especie}: pico no período '{periodo_pico}' | horas mais frequentes: {horas_pico}")

# Gráfico de linhas comparando as 3 espécies hora a hora
fig, ax = plt.subplots(figsize=(11, 4))
cores = {"Capivara": "#7a9e3b", "Sapo-cururu": "#3b7a9e", "Lobo-guará": "#9e3b7a"}
for especie in ESPECIES_ESCOLHIDAS:
    serie = df[df["especie"] == especie]["hora"].value_counts().sort_index()
    ax.plot(serie.index, serie.values, marker="o", label=especie, color=cores[especie])
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Quantidade de registros")
ax.set_title("Desafio 2 — Horários de atividade: Capivara x Sapo-cururu x Lobo-guará")
ax.legend()
plt.show()

### ✅ Resposta — Desafio 2

* **Visão geral:** o período com mais registros é a **Manhã (1.419)**, seguido de Tarde (1.317), Noite (1.177) e Madrugada (1.087). Por hora, os picos estão às **6h, 18h e 7h** — início da manhã e fim da tarde.
* **Capivara:** pico pela manhã (292 registros) e à tarde (283); horas mais frequentes: **13h, 9h e 6h** → atividade **diurna**.
* **Sapo-cururu:** pico na madrugada (250) e à noite (209); horas: **1h, 0h e 19h** → atividade **noturna**, típica de anfíbios (evitam calor e perda de água).
* **Lobo-guará:** manhã (133) e noite (114); horas: **18h, 6h e 16h** → atividade **crepuscular** (amanhecer e anoitecer).

**Conclusão:** a fauna é mais registrada **de manhã**, mas as espécies têm padrões claramente distintos — diurno (capivara), noturno (sapo-cururu) e crepuscular (lobo-guará). Para maximizar registros, o horário ideal depende da espécie-alvo.

## 💧 Desafio 3 — Vale a pena procurar anfíbios em períodos mais úmidos?

**Pergunta:** os anfíbios são registrados com maior frequência quando a umidade é mais alta?

**Abordagem:** dividimos os registros **com leitura de umidade** em 5 faixas (40–60% até 90–100%) e calculamos, em cada faixa, a **proporção de registros que são de anfíbios**. Usamos proporção (e não total absoluto) porque as faixas têm quantidades diferentes de registros. Complementamos comparando a umidade média dos registros de anfíbios com a dos demais grupos.

In [0]:
# ============================================================
# DESAFIO 3 — ANFÍBIOS x UMIDADE DO AR
# ============================================================
# Usamos apenas registros COM leitura de umidade (dropna descarta os
# 175 sem leitura — eles não têm a informação necessária p/ esta análise)
com_umidade = df.dropna(subset=["umidade_pct"]).copy()

# Dividimos a umidade em 5 faixas ('bins') para comparar grupos
FAIXAS = [40, 60, 70, 80, 90, 100]
ROTULOS = ["40-60%", "60-70%", "70-80%", "80-90%", "90-100%"]
com_umidade["faixa_umidade"] = pd.cut(com_umidade["umidade_pct"], bins=FAIXAS, labels=ROTULOS)

# Para cada faixa: total de registros, quantos são de anfíbios e a PROPORÇÃO
tabela_umidade = com_umidade.groupby("faixa_umidade", observed=True).agg(
    total_registros=("id_registro", "count"),
    registros_anfibios=("grupo", lambda grupos: (grupos == "Anfíbio").sum()))
tabela_umidade["pct_anfibios"] = (tabela_umidade["registros_anfibios"]
                                  / tabela_umidade["total_registros"] * 100).round(1)
display(tabela_umidade)

# Comparação complementar: umidade média nos registros de anfíbios x demais grupos
media_anfibios = com_umidade.loc[com_umidade["grupo"] == "Anfíbio", "umidade_pct"].mean()
media_outros = com_umidade.loc[com_umidade["grupo"] != "Anfíbio", "umidade_pct"].mean()
print(f"Umidade média nos registros de anfíbios: {media_anfibios:.1f}%")
print(f"Umidade média nos demais registros: {media_outros:.1f}%")

# Gráfico: proporção de anfíbios por faixa de umidade
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tabela_umidade.index.astype(str), tabela_umidade["pct_anfibios"], color="#3b7a9e")
ax.set_xlabel("Faixa de umidade relativa do ar")
ax.set_ylabel("% de registros que são de anfíbios")
ax.set_title("Desafio 3 — Proporção de anfíbios por faixa de umidade")
plt.show()

### ✅ Resposta — Desafio 3

**Sim — quanto maior a umidade, maior a proporção de anfíbios nos registros:**

| Faixa de umidade | % dos registros que são de anfíbios |
|---|---|
| 40–60% | 4,4% |
| 60–70% | 12,3% |
| 70–80% | 20,8% |
| 80–90% | 32,3% |
| 90–100% | 47,7% |

* A proporção de anfíbios **cresce de forma consistente** com a umidade: de ~4% nas faixas secas para quase metade dos registros na faixa mais úmida.
* A umidade média nos registros de anfíbios é **84,1%**, contra **74,5%** nos demais registros.

**Conclusão:** o padrão é claro e biologicamente coerente (anfíbios dependem de umidade para a pele e a reprodução). **Vale a pena planejar o monitoramento de anfíbios para períodos com umidade ≥ 80%**, priorizando madrugadas e noites — como reforça o Desafio 2.

## 📷 Desafio 4 — Onde concentrar o monitoramento?

**Pergunta:** quais câmeras têm maior quantidade de registros e maior variedade de espécies? Elas estão concentradas em alguma das três áreas do parque?

**Abordagem:** para cada câmera calculamos o **total de registros** (atividade) e o **número de espécies distintas** (variedade), junto com a área a que pertence (mapeamento do README: CAM01–04 Cerrado aberto, CAM05–08 Mata de galeria, CAM09–12 Veredas e áreas úmidas). Depois agregamos os totais por área.

In [0]:
# ============================================================
# DESAFIO 4 — QUAIS CÂMERAS CONCENTRAM A ATIVIDADE?
# ============================================================
# Para cada câmera calculamos:
# - registros: total de registros (ATIVIDADE)
# - especies: quantidade de espécies diferentes (VARIEDADE)
# agrupando por área e câmera, ordenado do mais ativo para o menos ativo
estatisticas_cameras = (df.groupby(["area", "id_camera"], as_index=False)
                        .agg(registros=("id_registro", "count"),
                             especies=("especie", "nunique"))
                        .sort_values("registros", ascending=False))
display(estatisticas_cameras)

# Resumo agregado por área do parque
resumo_area = df.groupby("area").agg(registros=("id_registro", "count"),
                                     especies=("especie", "nunique"))
print("Resumo por área do parque:")
display(resumo_area)

# Gráfico de dispersão: atividade (x) x variedade (y), com cor por área
fig, ax = plt.subplots(figsize=(9, 5))
cores_area = {"Cerrado aberto": "#c8a04a", "Mata de galeria": "#4a7a3b", "Veredas e áreas úmidas": "#4a7ac8"}
for area, grupo_cameras in estatisticas_cameras.groupby("area"):
    ax.scatter(grupo_cameras["registros"], grupo_cameras["especies"],
               label=area, color=cores_area[area], s=90)
for _, linha in estatisticas_cameras.iterrows():
    ax.annotate(linha["id_camera"], (linha["registros"], linha["especies"]),
                xytext=(4, 4), textcoords="offset points", fontsize=9)
ax.set_xlabel("Total de registros")
ax.set_ylabel("Espécies distintas registradas")
ax.set_title("Desafio 4 — Atividade x variedade por câmera")
ax.legend()
plt.show()

### ✅ Resposta — Desafio 4

**Câmeras com mais registros:**

1. **CAM09** — 523 registros (12 espécies) · Veredas e áreas úmidas
2. **CAM12** — 504 registros (13 espécies) · Veredas e áreas úmidas
3. **CAM10** — 474 registros (12 espécies) · Veredas e áreas úmidas
4. CAM02 — 471 registros (12 espécies) · Cerrado aberto
5. CAM04 — 442 registros (12 espécies) · Cerrado aberto

**Concentração por área:**

| Área | Registros | Espécies |
|---|---|---|
| Veredas e áreas úmidas | 1.817 | 13 |
| Cerrado aberto | 1.624 | 12 |
| Mata de galeria | 1.559 | 13 |

**Conclusão:** as três câmeras mais ativas (**CAM09, CAM12 e CAM10**) estão **todas concentradas na área de Veredas e áreas úmidas**, que lidera o parque em registros. A variedade de espécies é alta em praticamente todas as câmeras (12–13 espécies), então o diferencial entre elas é a atividade. **Recomendação: concentrar o esforço de monitoramento geral em Veredas e áreas úmidas, prioritariamente CAM09 e CAM12.**

## 🐆 Desafio 5 — Onde procurar a onça-pintada?

**Pergunta:** com base nos registros deste mês, em quais câmeras, áreas e períodos do dia você recomendaria concentrar o próximo monitoramento para registrar onça-pintada?

**Abordagem:** a onça-pintada é a espécie com menos registros do dataset (20 em 5.000). Isolamos todos os registros dela e analisamos em quais câmeras/áreas, períodos e condições ambientais (umidade e temperatura) ela apareceu.

In [0]:
# ============================================================
# DESAFIO 5 — ONDE PROCURAR A ONÇA-PINTADA?
# ============================================================
# Filtramos os registros da espécie — são raros, e por isso mesmo é
# importante direcionar bem o esforço de monitoramento
oncas = df[df["especie"] == "Onça-pintada"].copy()
print(f"Total de registros de onça-pintada em maio/2026: {len(oncas)}")

# 1) Em quais câmeras e áreas ela foi registrada?
print("\nRegistros por câmera e área:")
print(oncas.groupby(["id_camera", "area"]).size().sort_values(ascending=False))

# 2) Em quais períodos do dia?
print("\nRegistros por período do dia:")
print(oncas["periodo"].value_counts())

# 3) Detalhe: horas de todas as ocorrências (lista completa)
print("\nHoras dos registros:", sorted(oncas["hora"].tolist()))

# 4) Condições ambientais médias nesses registros
# (informamos também quantos registros tinham leitura disponível)
print(f"\nUmidade média nesses registros: {oncas['umidade_pct'].mean():.1f}%"
      f" (leitura disponível em {oncas['umidade_pct'].notna().sum()} de {len(oncas)})")
print(f"Temperatura média nesses registros: {oncas['temperatura_c'].mean():.1f} °C"
      f" (leitura disponível em {oncas['temperatura_c'].notna().sum()} de {len(oncas)})")

### ✅ Resposta — Desafio 5

Apenas **20 registros** em 30 dias — a espécie mais rara do dataset. O padrão é claro:

* **Câmeras/área:** 17 dos 20 registros (85%) ocorreram na **Mata de galeria** (CAM06: 6 registros, CAM08: 5, CAM07: 4, CAM05: 2); os demais em Veredas (CAM12: 2, CAM11: 1). Nenhum registro no Cerrado aberto.
* **Período:** 16 dos 20 registros (80%) foram à **noite (10) ou na madrugada (6)**; apenas 4 em horário diurno. As ocorrências se concentram entre **18h–23h e 0h–2h**.
* **Umidade:** média de **82,7%** nos registros com leitura disponível — ambientes úmidos, coerentes com a Mata de galeria.

**Recomendação para o próximo monitoramento:** concentrar o esforço nas câmeras da **Mata de galeria — prioritariamente CAM06 e CAM08** — durante a **noite e a madrugada (aprox. 18h às 6h)**. Como ponto secundário, manter a CAM12 (Veredas), que também registrou a espécie e é uma das câmeras mais ativas do parque.

## ⏱️ Bônus 2 — Quanto tempo duram os registros?

**Pergunta:** quais espécies apresentam os registros mais longos e mais curtos? Como a duração se distribui entre as espécies e o que isso pode indicar?

**Abordagem:** calculamos a **média** e a **mediana** da duração por espécie. A mediana complementa a média porque é menos sensível a registros extremamente longos, que puxam a média para cima. Acompanhamos também quantos registros cada espécie tem (amostras muito pequenas pedem cautela na generalização).

In [0]:
# ============================================================
# BÔNUS 2 — DURAÇÃO DOS REGISTROS POR ESPÉCIE
# ============================================================
# Para cada espécie: duração média, mediana e nº de registros.
# 'agg' com nomes nomeados cria as colunas já com os rótulos desejados.
duracao_por_especie = (df.groupby("especie")["duracao_segundos"]
                       .agg(media="mean", mediana="median", registros="count")
                       .round(1)
                       .sort_values("media", ascending=False))
display(duracao_por_especie)

print(f"Registros mais LONGOS em média: {duracao_por_especie['media'].idxmax()}"
      f" ({duracao_por_especie['media'].max():.1f} s)")
print(f"Registros mais CURTOS em média: {duracao_por_especie['media'].idxmin()}"
      f" ({duracao_por_especie['media'].min():.1f} s)")

# Gráfico de barras horizontais (maior no topo)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(duracao_por_especie.index, duracao_por_especie["media"], color="#8a6db1")
ax.set_xlabel("Duração média (segundos)")
ax.set_title("Bônus 2 — Duração média dos registros por espécie")
ax.invert_yaxis()
plt.show()

### ✅ Resposta — Bônus 2

* **Registros mais longos (média): onça-pintada (31,9 s)** — porém com apenas 20 registros, o que pede cautela na generalização — seguida de **cervo-do-pantanal (28,9 s)**.
* **Registros mais curtos (média): tatu-canastra (25,9 s)**.
* De modo geral, as durações médias ficam numa faixa estreita (≈ 26–29 s), com exceção da onça-pintada.

**Interpretação:** a pequena variação entre a maioria das espécies sugere que a duração depende mais do funcionamento da câmera (tempo de detecção) do que da espécie. As exceções são biologicamente plausíveis: um grande predador em deslocamento tende a permanecer mais tempo no campo de visão (registro mais longo), enquanto animais rápidos e discretos, como os tatus, geram registros curtos.

## 🐺 Bônus 3 — O comportamento do lobo-guará

**Pergunta:** em quais horários e em quais câmeras o lobo-guará foi mais registrado? Existe algum padrão que se destaque?

**Abordagem:** filtramos os 425 registros da espécie e analisamos a distribuição por hora, por período do dia, por câmera e por área do parque.

In [0]:
# ============================================================
# BÔNUS 3 — COMPORTAMENTO DO LOBO-GUARÁ
# ============================================================
lobos = df[df["especie"] == "Lobo-guará"].copy()
print(f"Registros de lobo-guará: {len(lobos)}")

# 1) Horários: distribuição por período e horas mais frequentes
print("\nRegistros por período do dia:")
print(lobos["periodo"].value_counts())
print("\n5 horas com mais registros:", list(lobos["hora"].value_counts().head(5).index))

# 2) Câmeras com mais registros (top 6)
print("\nRegistros por câmera (top 6):")
print(lobos["id_camera"].value_counts().head(6))

# 3) Áreas do parque
print("\nRegistros por área:")
print(lobos["area"].value_counts())

# Gráfico: atividade do lobo-guará por hora do dia
serie_lobo = lobos["hora"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(serie_lobo.index, serie_lobo.values, color="#9e5e3b")
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Quantidade de registros")
ax.set_title("Bônus 3 — Registros de lobo-guará por hora do dia")
plt.show()

### ✅ Resposta — Bônus 3

* **Horários:** atividade com dois picos — **18h (46 registros), 6h (42), 16h (39), 19h (38) e 7h (36)** — um padrão **crepuscular** bem definido (amanhecer e anoitecer), com a manhã à frente (133 registros), seguida da noite (114).
* **Câmeras/área:** as câmeras com mais registros são **CAM02 e CAM04 (71 cada), CAM01 (66) e CAM03 (45)** — todas do **Cerrado aberto**, que concentra 253 dos 425 registros (≈ 60%).

**Padrão destacável:** o lobo-guará do parque é um animal de **ambiente aberto com hábitos crepusculares**, circulando principalmente pelas câmeras CAM01–CAM04 no início da manhã e no fim da tarde — coerente com a espécie real, que movimenta-se por áreas abertas do Cerrado principalmente no crepúsculo.

## ❓ Bônus 4 — Minha própria investigação

**Pergunta escolhida:** *aves, mamíferos e anfíbios possuem picos de atividade em horários claramente distintos?*

**Motivação:** se cada grupo tem "agenda" própria, a equipe consegue planejar expedições específicas (ex.: anfíbios à noite, aves de dia) e escolher melhor quando deslocar equipes de campo e revisar câmeras.

**Abordagem:** cruzamos grupo × período do dia (tabela cruzada) e identificamos as horas mais frequentes de cada grupo.

In [0]:
# ============================================================
# BÔNUS 4 — ATIVIDADE POR GRUPO FAUNÍSTICO AO LONGO DO DIA
# ============================================================
# Tabela cruzada: registros de cada grupo em cada período do dia
# (reindex garante a ordem cronológica das colunas)
atividade_grupos = pd.crosstab(df["grupo"], df["periodo"])
atividade_grupos = atividade_grupos[["Madrugada", "Manhã", "Tarde", "Noite"]]
print("Registros por grupo em cada período:")
display(atividade_grupos)

# Horas mais frequentes de cada grupo
for grupo in ["Mamífero", "Ave", "Anfíbio"]:
    horas_top = list(df[df["grupo"] == grupo]["hora"].value_counts().head(3).index)
    print(f"{grupo} -> horas com mais registros: {horas_top}")

# Gráfico de linhas: atividade de cada grupo ao longo do dia
fig, ax = plt.subplots(figsize=(11, 4))
cores_grupo = {"Mamífero": "#7a9e3b", "Ave": "#e0a83b", "Anfíbio": "#3b7a9e"}
for grupo in ["Mamífero", "Ave", "Anfíbio"]:
    serie = df[df["grupo"] == grupo]["hora"].value_counts().sort_index()
    ax.plot(serie.index, serie.values, marker="o", label=grupo, color=cores_grupo[grupo])
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Quantidade de registros")
ax.set_title("Bônus 4 — Atividade ao longo do dia por grupo faunístico")
ax.legend()
plt.show()

### ✅ Resposta — Bônus 4

**Sim — os três grupos têm agendas claramente distintas:**

| Grupo | Picos de atividade | Horas mais frequentes |
|---|---|---|
| Mamíferos | Manhã (868) e tarde (773) | 6h, 7h, 16h — diurno com pico matinal |
| Aves | Manhã (426) e tarde (424) | 15h, 7h, 10h — estritamente diurno |
| Anfíbios | Madrugada (516) e noite (509) | 1h, 19h, 3h — noturno |

* **Aves** praticamente desaparecem no escuro: 207 registros à noite/madrugada contra 850 em período diurno.
* **Anfíbios** fazem o oposto: 1.025 registros no escuro contra apenas 245 de dia.
* **Mamíferos** são intermediários: predominam de dia, mas mantêm atividade noturna relevante (569 registros à noite + 463 na madrugada).

**Conclusão:** cada grupo tem janela de atividade própria, e o monitoramento pode ser **direcionado por horário**: aves e mamíferos de manhã/tarde; anfíbios à noite e na madrugada — idealmente em condições úmidas, como mostrou o Desafio 3.

## 🎯 Conclusões gerais para a apresentação

1. **Tratamento dos dados:** temperatura e umidade vieram como texto (ex.: `"19.3 °C"`) e foram convertidas para número; datas convertidas para `datetime`; 247 registros sem temperatura e 175 sem umidade foram mantidos como ausentes (sem imputação), e cada análise usou apenas os registros com a leitura necessária disponível. Não havia duplicatas nem categorias fora do esperado.
2. **Capivaras vivem em grupos:** média de ≈ **6 indivíduos por registro**; 99,2% dos registros têm 2 ou mais indivíduos.
3. **A fauna é mais ativa de manhã** (1.419 registros), mas cada espécie tem seu padrão: capivara diurna, sapo-cururu noturno, lobo-guará crepuscular.
4. **Anfíbios e umidade caminham juntos:** a proporção de anfíbios sobe de 4,4% (umidade 40–60%) para 47,7% (90–100%). Monitorar anfíbios em noites/madrugadas úmidas.
5. **Veredas e áreas úmidas é a área mais ativa** (1.817 registros; câmeras CAM09, CAM12 e CAM10 lideram) — onde concentrar o monitoramento geral.
6. **Onça-pintada:** rara (20 registros), noturna e associada à Mata de galeria (85% dos casos). Priorizar CAM06 e CAM08 entre ~18h e 6h.
7. **Duração dos registros** varia pouco entre espécies (≈ 26–29 s), com exceção da onça-pintada (média 31,9 s, amostra pequena) — sugerindo que a duração depende mais da câmera que da espécie.
8. **Lobo-guará:** crepuscular e típico do Cerrado aberto (CAM01–CAM04, ≈ 60% dos registros da espécie).
9. **Cada grupo faunístico tem sua "agenda":** aves e mamíferos de dia, anfíbios à noite — o esforço de campo pode ser planejado por horário e por área-alvo.